# DreamBrush 3DGS Training

This notebook turns a DreamBrush `CaptureBundle` into a Nerfstudio-ready dataset, trains a 3D Gaussian Splat (Splatfacto), and exports a PLY for the iOS viewer.

**Pipeline**
1. Download bundle from Google Drive
2. Convert to `transforms.json` + `images/`
3. Build `sparse_pc.ply` from LiDAR depth for better Splatfacto initialization
4. Train (`ns-train splatfacto`)
5. Export PLY (`ns-export gaussian-splat`)

> Note: The app exports camera transforms in **row-major** layout (as required by Nerfstudio), so this notebook does not transpose matrices.

## 0) Environment setup (GPU VM)

This workflow expects a CUDA-capable GPU for training. Run this cell once per VM:

In [ ]:
import os, sys, subprocess, venv, textwrap, importlib

# --- config ---
VENV_DIR = ".venv"
REQS = ["nerfstudio", "gdown", "imageio", "pillow", "tqdm", "matplotlib", "open3d", "pyyaml"]
# --------------

def run(cmd, **kwargs):
    print(">>", " ".join(cmd))
    subprocess.run(cmd, check=True, **kwargs)

# 1) Create venv that can SEE the preinstalled system packages (incl. torch/cuda)
if not os.path.isdir(VENV_DIR):
    print(f"Creating venv at {VENV_DIR} (with system-site-packages)...")
    venv.EnvBuilder(with_pip=True, system_site_packages=True).create(VENV_DIR)
else:
    print(f"Venv already exists at {VENV_DIR}")

vpy = os.path.join(VENV_DIR, "bin", "python")
if not os.path.exists(vpy):
    raise RuntimeError(f"Expected venv python at {vpy}, but it wasn't found.")

# 2) Tooling
run([vpy, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])

# 3) Fix the common Ubuntu/apt "distutils blinker 1.4 can't uninstall" issue:
#    shadow it inside the venv instead of trying to remove the system one.
run([vpy, "-m", "pip", "install", "--ignore-installed", "blinker>=1.6"])

# 4) Pin the *system* torch stack so pip won't download/replace it
constraints_path = "/tmp/torch-constraints.txt"
pin_code = r"""
import importlib.metadata as md
pins=[]
for name in ("torch","torchvision","torchaudio"):
    try:
        pins.append(f"{name}=={md.version(name)}")
    except md.PackageNotFoundError:
        pass
path = """ + repr(constraints_path) + r"""
with open(path,"w") as f:
    f.write("\n".join(pins) + ("\n" if pins else ""))
print(path)
print(open(path).read() if pins else "(no torch packages found to pin)")
"""
out = subprocess.check_output([vpy, "-c", pin_code], text=True)
print("Torch constraints:\n" + out)

# 5) Install your deps, respecting the torch constraints (so no slow torch install)
run([vpy, "-m", "pip", "install", "-c", constraints_path, *REQS])

# 6) Make this venv usable as a Jupyter kernel (optional but recommended)
run([vpy, "-m", "pip", "install", "-c", constraints_path, "ipykernel"])
run([vpy, "-m", "ipykernel", "install", "--user", "--name", "runpod-nerf", "--display-name", "Python (runpod-nerf)"])

# 7) Also make venv packages importable *in the current kernel* (no restart needed)
#    (This is a pragmatic hack: we add the venv site-packages to sys.path.)
site_pkgs = subprocess.check_output([vpy, "-c", "import site; print(site.getsitepackages()[0])"], text=True).strip()
if site_pkgs not in sys.path:
    sys.path.insert(0, site_pkgs)
importlib.invalidate_caches()

os.environ["VIRTUAL_ENV"] = os.path.abspath(VENV_DIR)
os.environ["PATH"] = os.path.abspath(os.path.join(VENV_DIR, "bin")) + os.pathsep + os.environ.get("PATH", "")

print("\n✅ Done.")
print("Best practice: switch your notebook kernel to 'Python (runpod-nerf)' for a clean env.")
print("But you can also keep going in this kernel now; venv site-packages were added to sys.path.")

## 1) Imports + configuration

Fill in your Google Drive link and choose output paths.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import shutil
import zipfile
from pathlib import Path

import numpy as np
import imageio.v2 as imageio
from PIL import Image
from tqdm import tqdm

# ---- User config ----
GDRIVE_URL = "https://drive.google.com/file/d/1qmHIn1vPP_IwXCPBjCV7DCoyk7bQeYqO/view?usp=drive_link"

WORK_DIR = WORK_DIR = Path.cwd() / "3dgs_train"
DOWNLOAD_DIR = WORK_DIR / "downloads"
EXTRACT_DIR = WORK_DIR / "bundles"
DATASET_DIR = WORK_DIR / "nerfstudio_dataset"
EXPORT_DIR = WORK_DIR / "exports"

USE_KEYFRAMES = True
FRAME_STRIDE = 1          # keep every Nth frame
MAX_FRAMES = None         # cap number of frames (e.g., 150)

DEPTH_STRIDE = 4          # downsample depth pixels (higher = fewer points)
MIN_DEPTH_M = 0.2
MAX_DEPTH_M = 8.0
VOXEL_SIZE_M = 0.02       # 2 cm voxel size for sparse point cloud
MAX_POINTS_PER_FRAME = 200_000

# If you see mirrored results, try enabling this camera-axis conversion.
# This flips the camera Z axis in camera coordinates before writing transforms.
APPLY_CAMERA_AXIS_CONVERSION = False
CAMERA_AXIS_CONVERSION = np.diag([1.0, 1.0, -1.0, 1.0]).astype(np.float32)

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

## 2) Download + extract bundle from Google Drive

In [ ]:
import importlib


def extract_gdrive_file_id(url: str) -> str | None:
    if not url:
        return None
    # Patterns: .../file/d/<ID>/view or id=<ID>
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
    if match:
        return match.group(1)
    match = re.search(r"id=([a-zA-Z0-9_-]+)", url)
    if match:
        return match.group(1)
    return None


def download_from_gdrive(url: str, out_path: Path) -> Path:
    if out_path.exists():
        return out_path
    file_id = extract_gdrive_file_id(url)
    if not file_id:
        raise ValueError("Could not extract file id from GDRIVE_URL. Use a share link or '?id=' URL.")

    gdown = importlib.import_module("gdown")
    download_url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(download_url, str(out_path), quiet=False)
    if not out_path.exists():
        raise RuntimeError("Download failed; file not found after gdown.")
    return out_path


zip_path = DOWNLOAD_DIR / "capture_bundle.zip"
if GDRIVE_URL:
    zip_path = download_from_gdrive(GDRIVE_URL, zip_path)
else:
    raise ValueError("Set GDRIVE_URL before running this cell.")

# Extract
bundle_root = EXTRACT_DIR / zip_path.stem
if not bundle_root.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(bundle_root)

# Find CaptureBundle_* folder
bundle_dirs = [p for p in bundle_root.rglob("CaptureBundle_*") if p.is_dir()]

non_macos = [p for p in bundle_dirs if "__MACOSX" not in p.parts]
if not bundle_dirs:
    raise FileNotFoundError("No CaptureBundle_* folder found in extracted archive")

bundle_dir = non_macos[0] if non_macos else bundle_dirs[0]
print("Found bundle:", bundle_dir)

## 3) Inspect bundle + load anchors

In [ ]:
manifest = json.loads((bundle_dir / "manifest.json").read_text())
anchors = json.loads((bundle_dir / "anchors.json").read_text())

print("Bundle ID:", manifest.get("bundleId"))
print("Frame count:", manifest.get("captureStats", {}).get("frameCount"))
print("Keyframe count:", manifest.get("captureStats", {}).get("keyframeCount"))
print("Depth enabled:", manifest.get("captureSettings", {}).get("depthEnabled"))
print("Coordinate conventions:", manifest.get("coordinateConventions", {}))

layout = manifest.get("coordinateConventions", {}).get("matrixLayout")
if layout and layout != "row_major":
    raise ValueError(f"This bundle uses matrixLayout={layout}. Re-export with row_major to avoid transpose in training.")

root_anchor = anchors.get("rootAnchor", {})
root_transform = root_anchor.get("transform")
if not root_transform:
    raise ValueError("anchors.json missing rootAnchor.transform")

# Row-major 4x4
T_root_world = np.array(root_transform, dtype=np.float32)
print("Root anchor transform (row-major): ", T_root_world)

## 4) Select frames

Uses keyframes by default.

In [ ]:
keyframes_dir = bundle_dir / "keyframes"
frames_meta_dir = bundle_dir / "frames" / "meta"
frames_rgb_dir = bundle_dir / "frames" / "rgb"
frames_depth_dir = bundle_dir / "frames" / "depth"

if USE_KEYFRAMES:
    frame_ids = sorted([int(p.stem) for p in keyframes_dir.glob("*.jpg")])
else:
    frame_ids = sorted([int(p.stem) for p in frames_rgb_dir.glob("*.jpg")])

# Apply stride and max
frame_ids = frame_ids[::FRAME_STRIDE]
if MAX_FRAMES:
    frame_ids = frame_ids[:MAX_FRAMES]

print(f"Selected {len(frame_ids)} frames")

## 5) Build Nerfstudio dataset (images + transforms.json)

In [ ]:
from datetime import datetime


def load_frame_meta(frame_id: int) -> dict:
    meta_path = frames_meta_dir / f"{frame_id:06d}.json"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing meta for frame {frame_id}: {meta_path}")
    return json.loads(meta_path.read_text())


def rows_to_matrix(rows: list[list[float]]) -> np.ndarray:
    return np.array(rows, dtype=np.float32)


def intrinsics_from_meta(meta: dict):
    intr = meta["camera"]["intrinsics"]
    fx = intr[0][0]
    fy = intr[1][1]
    cx = intr[0][2]
    cy = intr[1][2]
    w = meta["camera"]["imageResolution"]["width"]
    h = meta["camera"]["imageResolution"]["height"]
    return fx, fy, cx, cy, w, h


# Dataset target
dataset_name = bundle_dir.name
dataset_dir = DATASET_DIR / dataset_name
images_dir = dataset_dir / "images"
images_dir.mkdir(parents=True, exist_ok=True)

frames = []
intrinsics_list = []

for frame_id in tqdm(frame_ids, desc="Processing frames"):
    meta = load_frame_meta(frame_id)
    fx, fy, cx, cy, w, h = intrinsics_from_meta(meta)
    intrinsics_list.append([fx, fy, cx, cy, w, h])

    # Copy image
    src_img = keyframes_dir / f"{frame_id:06d}.jpg" if USE_KEYFRAMES else frames_rgb_dir / f"{frame_id:06d}.jpg"
    dst_img = images_dir / f"{frame_id:06d}.jpg"
    if not dst_img.exists():
        shutil.copy2(src_img, dst_img)

    # Camera transform: row-major
    T_cam_world = rows_to_matrix(meta["camera"]["transform"])
    T_cam_anchor = np.linalg.inv(T_root_world) @ T_cam_world
    if APPLY_CAMERA_AXIS_CONVERSION:
        T_cam_anchor = T_cam_anchor @ CAMERA_AXIS_CONVERSION

    frames.append({
        "file_path": f"images/{frame_id:06d}.jpg",
        "transform_matrix": T_cam_anchor.tolist()
    })

# Check intrinsics consistency
intrinsics_arr = np.array(intrinsics_list)
max_spread = intrinsics_arr.max(axis=0) - intrinsics_arr.min(axis=0)
print("Intrinsics max spread [fx, fy, cx, cy, w, h]:", max_spread)

fx, fy, cx, cy, w, h = intrinsics_list[0]

transforms = {
    "camera_model": "OPENCV",
    "fl_x": float(fx),
    "fl_y": float(fy),
    "cx": float(cx),
    "cy": float(cy),
    "w": int(w),
    "h": int(h),
    "ply_file_path": "sparse_pc.ply",
    "frames": frames
}

(dataset_dir / "transforms.json").write_text(json.dumps(transforms, indent=2))
print("Wrote transforms.json to", dataset_dir)

## 6) Build `sparse_pc.ply` from depth

This gives Splatfacto a strong geometric initialization.

In [ ]:
def load_depth_m(path: Path) -> np.ndarray:
    depth = imageio.imread(path)
    if depth.dtype != np.uint16:
        depth = depth.astype(np.uint16)
    return depth.astype(np.float32) / 1000.0  # mm -> meters


def unproject_depth(depth_m: np.ndarray, fx: float, fy: float, cx: float, cy: float, stride: int):
    h, w = depth_m.shape
    us = np.arange(0, w, stride)
    vs = np.arange(0, h, stride)
    uu, vv = np.meshgrid(us, vs)
    z = depth_m[np.ix_(vs, us)]
    mask = (z > MIN_DEPTH_M) & (z < MAX_DEPTH_M)

    u = uu[mask]
    v = vv[mask]
    z = z[mask]

    x = (u - cx) / fx * z
    y = (v - cy) / fy * z
    pts = np.stack([x, y, z], axis=1).astype(np.float32)
    return pts, (u, v)


def voxel_downsample(points: np.ndarray, colors: np.ndarray | None, voxel_size: float):
    if voxel_size <= 0:
        return points, colors
    voxel = np.floor(points / voxel_size).astype(np.int32)
    _, unique_idx = np.unique(voxel, axis=0, return_index=True)
    if colors is None:
        return points[unique_idx], None
    return points[unique_idx], colors[unique_idx]


def write_ply_ascii(path: Path, points: np.ndarray, colors: np.ndarray | None = None):
    with path.open("w", encoding="ascii") as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {len(points)}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        if colors is not None:
            f.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
        f.write("end_header\n")
        if colors is None:
            for x, y, z in points:
                f.write(f"{x} {y} {z}\n")
        else:
            for (x, y, z), (r, g, b) in zip(points, colors):
                f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")
all_points = []
all_colors = []

for frame_id in tqdm(frame_ids, desc="Depth to points"):
    meta = load_frame_meta(frame_id)
    depth_meta = meta.get("depth") or {}
    if not depth_meta.get("available"):
        continue

    depth_path = frames_depth_dir / f"{frame_id:06d}.png"
    if not depth_path.exists():
        continue

    depth_m = load_depth_m(depth_path)
    # Scale intrinsics to depth resolution
    fx, fy, cx, cy, w, h = intrinsics_from_meta(meta)
    depth_h, depth_w = depth_m.shape
    fx_d = fx * (depth_w / w)
    fy_d = fy * (depth_h / h)
    cx_d = cx * (depth_w / w)
    cy_d = cy * (depth_h / h)

    pts_cam, (u_depth, v_depth) = unproject_depth(depth_m, fx_d, fy_d, cx_d, cy_d, DEPTH_STRIDE)
    if pts_cam.shape[0] == 0:
        continue

    # Transform to anchor space
    T_cam_world = rows_to_matrix(meta["camera"]["transform"])
    T_cam_anchor = np.linalg.inv(T_root_world) @ T_cam_world
    if APPLY_CAMERA_AXIS_CONVERSION:
        T_cam_anchor = T_cam_anchor @ CAMERA_AXIS_CONVERSION

    ones = np.ones((pts_cam.shape[0], 1), dtype=np.float32)
    pts_h = np.concatenate([pts_cam, ones], axis=1)
    pts_anchor = (T_cam_anchor @ pts_h.T).T[:, :3]

    # Optional colors sampled from RGB
    img_path = keyframes_dir / f"{frame_id:06d}.jpg" if USE_KEYFRAMES else frames_rgb_dir / f"{frame_id:06d}.jpg"
    rgb = np.array(Image.open(img_path).convert("RGB"))
    rgb_h, rgb_w, _ = rgb.shape

    u_rgb = np.clip((u_depth * (rgb_w / depth_w)).astype(np.int32), 0, rgb_w - 1)
    v_rgb = np.clip((v_depth * (rgb_h / depth_h)).astype(np.int32), 0, rgb_h - 1)
    colors = rgb[v_rgb, u_rgb]

    if MAX_POINTS_PER_FRAME and pts_anchor.shape[0] > MAX_POINTS_PER_FRAME:
        idx = np.random.choice(pts_anchor.shape[0], MAX_POINTS_PER_FRAME, replace=False)
        pts_anchor = pts_anchor[idx]
        colors = colors[idx]

    all_points.append(pts_anchor)
    all_colors.append(colors)

if not all_points:
    raise RuntimeError("No depth points generated. Check depth availability and paths.")

points = np.concatenate(all_points, axis=0)
colors = np.concatenate(all_colors, axis=0)
print("Raw points:", points.shape)

points, colors = voxel_downsample(points, colors, VOXEL_SIZE_M)
print("Downsampled points:", points.shape)

sparse_path = dataset_dir / "sparse_pc.ply"
write_ply_ascii(sparse_path, points, colors)
print("Wrote sparse point cloud:", sparse_path)

## 7) Sanity checks (optional)

In [ ]:
# Basic bbox sanity check
mins = points.min(axis=0)
maxs = points.max(axis=0)
print("Point cloud bounds (m):")
print("  min:", mins)
print("  max:", maxs)

# Quick 2D scatter preview (top-down XZ)
import matplotlib.pyplot as plt

sample = points[np.random.choice(points.shape[0], min(5000, points.shape[0]), replace=False)]
plt.figure(figsize=(6, 6))
plt.scatter(sample[:, 0], sample[:, 2], s=1)
plt.title("Top-down view (X-Z)")
plt.axis("equal")
plt.show()

## 7.5) Preflight checks

Make sure the dataset and sparse point cloud are ready before training.


In [ ]:
import json
from pathlib import Path
import numpy as np

transforms_path = dataset_dir / "transforms.json"
if not transforms_path.exists():
    raise FileNotFoundError(f"Missing transforms.json at {transforms_path}")

transforms = json.loads(transforms_path.read_text())
frames = transforms.get("frames", [])
if not frames:
    raise ValueError("transforms.json has no frames")

ply_rel = transforms.get("ply_file_path")
if not ply_rel:
    raise ValueError("transforms.json missing ply_file_path for sparse init")
ply_path = dataset_dir / ply_rel
if not ply_path.exists():
    raise FileNotFoundError(f"Sparse point cloud not found: {ply_path}")

# Spot-check image files
missing = []
for frame in frames[:5]:
    img = dataset_dir / frame["file_path"]
    if not img.exists():
        missing.append(str(img))
if missing:
    raise FileNotFoundError(f"Missing image files: {missing}")

# Spot-check transform matrices
for frame in frames[:3]:
    mat = np.array(frame.get("transform_matrix"))
    if mat.shape != (4, 4):
        raise ValueError(f"transform_matrix wrong shape: {mat.shape}")
    if not np.isfinite(mat).all():
        raise ValueError("transform_matrix has NaNs/Infs")

print(f"Preflight OK: {len(frames)} frames, sparse_pc.ply present, sample images OK")


## 8) Train Splatfacto (Nerfstudio)

Run training on the GPU VM. This command will create an output folder under `outputs/`.

In [ ]:
# Example training command (edit as needed)
# Note: dataparser args come after the dataparser subcommand (nerfstudio-data).
!ns-train splatfacto --vis tensorboard --logging.steps-per-log 500 --logging.local-writer.no-enable nerfstudio-data --data "{dataset_dir}" --load-3D-points True


## 9) Export Gaussian Splat PLY

After training completes, export the PLY. Replace `CONFIG_PATH` with the path to the generated `config.yml` in `outputs/`.

In [ ]:
# Auto-detect latest Nerfstudio run and export PLY
outputs_root = Path("outputs")
config_paths = list(outputs_root.rglob("config.yml"))
if not config_paths:
    raise FileNotFoundError("No config.yml found under outputs/. Run training first.")

latest_config = max(config_paths, key=lambda p: p.stat().st_mtime)
CONFIG_PATH = latest_config
print("Using config:", CONFIG_PATH)

OUT_DIR = EXPORT_DIR / dataset_dir.name
OUT_DIR.mkdir(parents=True, exist_ok=True)

!ns-export gaussian-splat --load-config "{CONFIG_PATH}" --output-dir "{OUT_DIR}"

## 9.5) Locate exported PLY

Lists exported PLY files and sizes so you can download the right one.


In [ ]:
from pathlib import Path

ply_files = sorted(OUT_DIR.glob("*.ply"))
if not ply_files:
    raise FileNotFoundError(f"No .ply files found in {OUT_DIR}")

print(f"Export directory: {OUT_DIR}")
for p in ply_files:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"- {p.name} ({size_mb:.2f} MB)")


## 10) Alignment metadata (model → capture anchor)

Nerfstudio applies an orientation/centering transform and a scale during training. We invert those to align the exported PLY back into your capture anchor space.


In [ ]:
# Alignment from dataparser_transforms.json (no YAML parsing needed)
outputs_root = Path("outputs")
transform_paths = list(outputs_root.rglob("dataparser_transforms.json"))
if not transform_paths:
    raise FileNotFoundError("No dataparser_transforms.json found under outputs/. Run training first.")

latest_transform = max(transform_paths, key=lambda p: p.stat().st_mtime)
RUN_DIR = latest_transform.parent
print("Using run:", RUN_DIR)

# Where the exported PLY lives
OUT_DIR = EXPORT_DIR / dataset_dir.name
OUT_DIR.mkdir(parents=True, exist_ok=True)

payload = json.loads(latest_transform.read_text())
transform = payload.get("transform")
scale = float(payload.get("scale", 1.0))

if transform is None:
    raise ValueError("dataparser_transforms.json missing 'transform'")

T = np.array(transform, dtype=np.float32)
if T.shape == (3, 4):
    T4 = np.eye(4, dtype=np.float32)
    T4[:3, :4] = T
elif T.shape == (4, 4):
    T4 = T
else:
    raise ValueError(f"Unexpected transform shape: {T.shape}")

inv_T = np.linalg.inv(T4)
S_inv = np.diag([1.0 / scale, 1.0 / scale, 1.0 / scale, 1.0]).astype(np.float32)

# model (dataparser space) -> capture anchor space
model_to_anchor = inv_T @ S_inv

alignment = {
    "model_to_anchor_4x4": model_to_anchor.tolist(),
    "matrixLayout": "row_major"
}

alignment_path = OUT_DIR / "alignment.json"
alignment_path.write_text(json.dumps(alignment, indent=2))
print("Wrote", alignment_path)